# 05 · P1 确定性商业层 — Freight / Customs / Incoterms → Landed Cost

费率全部来自 `config/commercial.yaml`（商业真相），LLM 不参与数字。
计费重 = max(实重, 体积重)；Incoterm 决定卖方承担项 → seller_quote_price；landed_cost = 买方真实到手成本。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from bootstrap import build_controller, status
from services.config import load_settings, load_policy
print('trunk root:', ROOT)

In [ ]:
from services.commercial import compute_commercial, resolve_region
from services.config import load_commercial
CFG = load_commercial(ROOT)
quote = {'final_price': 9413.3, 'unit_price': 222.8, 'lead_time_days': 3, 'profit': 2172.3}
rfq = {'material':'6061','quantity':50,'weight_kg':0.5,'dimensions_mm':[100,50,10],'tolerance_grade':'IT7'}
for inco in ['EXW','FOB','CIF','DDP']:
    r = compute_commercial(quote, rfq, CFG, destination_country='US', shipping_mode='air', incoterm=inco)
    print('%-4s seller_quote=%9.2f landed=%9.2f duty=%7.2f freight=%7.2f lead=%2dd' % (
        inco, r['seller_quote_price'], r['landed_cost'], r['breakdown']['duty'], r['breakdown']['freight'], r['lead_time']['total_days']))

## 计费重: 致密件(实重占优) vs 抛货(体积重占优)

In [ ]:
dense = compute_commercial(quote, rfq, CFG, destination_country='US', shipping_mode='air')
bulky = compute_commercial(quote, {'material':'6061','quantity':10,'weight_kg':0.1,'dimensions_mm':[500,400,300]}, CFG, destination_country='US', shipping_mode='air')
print('致密件 chargeable=', dense['weight']['chargeable_kg'], '(actual', dense['weight']['total_actual_kg'], 'vs vol', dense['weight']['volumetric_kg'], ')')
print('抛货   chargeable=', bulky['weight']['chargeable_kg'], '(actual', bulky['weight']['total_actual_kg'], 'vs vol', bulky['weight']['volumetric_kg'], ')')

## 黄金链中的商业层 (端到端, 真实引擎报价 → landed cost)

In [ ]:
ctrl = build_controller()
r = ctrl.run(email_text='50 pcs 6061 aluminum brackets, 100x50x10mm, anodizing, IT7.',
             customer={'name':'Northwind','country':'US'}, destination_country='US', shipping_mode='air', incoterm='DDP')
cm = r['commercial']
print('state=', r['state'], 'incoterm=', cm['incoterm'], 'region=', cm['region'])
print('breakdown=', json.dumps(cm['breakdown'], ensure_ascii=False))
print('landed_cost=', cm['landed_cost'], 'seller_quote=', cm['seller_quote_price'], 'total_lead=', cm['total_lead_time_days'])